In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
!pip install transformers datasets accelerate -q

from datasets import load_dataset
wiki_dataset = load_dataset("wikitext", "wikitext-2-raw-v1")

README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

In [2]:
from transformers import GPT2Tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128)

wiki_train_raw = wiki_dataset["train"].filter(lambda x: len(x["text"].strip()) > 20)
wiki_val_raw = wiki_dataset["validation"].filter(lambda x: len(x["text"].strip()) > 20)

wiki_train = wiki_train_raw.select(range(min(20000, len(wiki_train_raw))))
wiki_val = wiki_val_raw.select(range(min(2000, len(wiki_val_raw))))

wiki_tokenized_train = wiki_train.map(tokenize_fn, batched=True, remove_columns=["text"])
wiki_tokenized_val = wiki_val.map(tokenize_fn, batched=True, remove_columns=["text"])

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Filter:   0%|          | 0/36718 [00:00<?, ? examples/s]

Filter:   0%|          | 0/3760 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [3]:
from transformers import DataCollatorForLanguageModeling
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [4]:
from transformers import GPT2Config, GPT2LMHeadModel

tiny_config = GPT2Config(vocab_size=tokenizer.vocab_size, n_positions=128, n_embd=128, n_layer=4, n_head=4)
zoom_config = GPT2Config(vocab_size=tokenizer.vocab_size, n_positions=128, n_embd=144, n_layer=4, n_head=4)
xsmall_config = GPT2Config(vocab_size=tokenizer.vocab_size, n_positions=128, n_embd=160, n_layer=4, n_head=4)
small_config = GPT2Config(vocab_size=tokenizer.vocab_size, n_positions=128, n_embd=192, n_layer=6, n_head=6)
medium_config = GPT2Config(vocab_size=tokenizer.vocab_size, n_positions=128, n_embd=256, n_layer=8, n_head=8)

In [5]:
from transformers import AutoModelForCausalLM

teacher_neo = AutoModelForCausalLM.from_pretrained("EleutherAI/gpt-neo-125m")
teacher_neo.eval()
teacher_neo.to("cuda")
print(f"GPT-Neo teacher params: {sum(p.numel() for p in teacher_neo.parameters())/1e6:.1f}M")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/526M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

GPTNeoForCausalLM LOAD REPORT from: EleutherAI/gpt-neo-125m
Key                                                   | Status     |  | 
------------------------------------------------------+------------+--+-
transformer.h.{0...11}.attn.attention.masked_bias     | UNEXPECTED |  | 
transformer.h.{0, 2, 4, 6, 8, 10}.attn.attention.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

GPT-Neo teacher params: 125.2M


In [6]:
import torch
import torch.nn.functional as F
from transformers import Trainer, TrainingArguments

class DistillationTrainer(Trainer):
    def __init__(self, teacher_model=None, temperature=2.0, alpha=0.5, **kwargs):
        super().__init__(**kwargs)
        self.teacher = teacher_model
        self.temperature = temperature
        self.alpha = alpha

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        outputs = model(**inputs)
        student_loss = outputs.loss
        student_logits = outputs.logits
        with torch.no_grad():
            teacher_outputs = self.teacher(**inputs)
            teacher_logits = teacher_outputs.logits
        T = self.temperature
        student_logits_flat = student_logits.view(-1, student_logits.size(-1))
        teacher_logits_flat = teacher_logits.view(-1, teacher_logits.size(-1))
        distill_loss = F.kl_div(
            F.log_softmax(student_logits_flat / T, dim=-1),
            F.softmax(teacher_logits_flat / T, dim=-1),
            reduction="batchmean"
        ) * (T ** 2)
        loss = self.alpha * student_loss + (1 - self.alpha) * distill_loss
        return (loss, outputs) if return_outputs else loss

def run_baseline_condition(config, seed, args_template):
    torch.manual_seed(seed)
    model = GPT2LMHeadModel(config)
    model.to("cuda")
    trainer = Trainer(
        model=model, args=args_template,
        train_dataset=wiki_tokenized_train, eval_dataset=wiki_tokenized_val, data_collator=data_collator,
    )
    trainer.train()
    result = trainer.evaluate()
    return result["eval_loss"]

def run_distilled_condition(config, teacher_model, seed, args_template, alpha=0.5, temperature=2.0):
    torch.manual_seed(seed)
    model = GPT2LMHeadModel(config)
    model.to("cuda")
    trainer = DistillationTrainer(
        model=model, teacher_model=teacher_model, temperature=temperature, alpha=alpha,
        args=args_template,
        train_dataset=wiki_tokenized_train, eval_dataset=wiki_tokenized_val, data_collator=data_collator,
    )
    trainer.train()
    plain_eval_trainer = Trainer(
        model=model, args=args_template,
        eval_dataset=wiki_tokenized_val, data_collator=data_collator,
    )
    result = plain_eval_trainer.evaluate()
    return result["eval_loss"]

In [7]:
wiki_neo_results = {}
configs = {"Tiny": tiny_config, "Medium": medium_config}

for name, cfg in configs.items():
    args = TrainingArguments(
        output_dir=f"./wiki_neo_{name}",
        per_device_train_batch_size=16, per_device_eval_batch_size=16,
        num_train_epochs=1, logging_steps=200, eval_strategy="steps",
        eval_steps=200, save_strategy="no", report_to="none",
    )
    base_loss = run_baseline_condition(cfg, seed=42, args_template=args)
    dist_loss = run_distilled_condition(cfg, teacher_neo, seed=42, args_template=args, alpha=0.5)
    wiki_neo_results[name] = {"baseline": base_loss, "distilled": dist_loss, "gap": base_loss - dist_loss}
    print(f"{name}, WikiText-2, GPT-Neo teacher: baseline={base_loss:.3f}, distilled={dist_loss:.3f}, gap={base_loss-dist_loss:.3f}")

print(wiki_neo_results)

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss
200,9.886469,9.245744
400,8.869641,8.481949
600,8.263844,8.018905
800,7.906105,7.747368
1000,7.702423,7.608018
1200,7.596097,7.555820


Step,Training Loss,Validation Loss
200,7.695747,7.114902
400,6.840396,6.511844
600,6.385471,6.155103
800,6.109517,5.943975
1000,5.948796,5.832542
1200,5.857100,5.787007


Tiny, WikiText-2, GPT-Neo teacher: baseline=7.554, distilled=7.699, gap=-0.145


Step,Training Loss,Validation Loss
200,9.132823,8.073412
400,7.618970,7.240476
600,7.148811,7.033104
800,7.028224,6.949170
1000,6.967133,6.903962
1200,6.914082,6.882484


Step,Training Loss,Validation Loss
200,7.132200,6.267690
400,5.943842,5.608276
600,5.534152,5.363917
800,5.372527,5.259959
1000,5.291503,5.203540
1200,5.239161,5.180930


Medium, WikiText-2, GPT-Neo teacher: baseline=6.882, distilled=6.992, gap=-0.110
{'Tiny': {'baseline': 7.553736686706543, 'distilled': 7.698877811431885, 'gap': -0.1451411247253418}, 'Medium': {'baseline': 6.881779193878174, 'distilled': 6.992213726043701, 'gap': -0.11043453216552734}}


In [8]:
print(f"{'Size':<10}{'Teacher':<12}{'Dataset':<14}{'Gap':<10}")
print(f"{'Tiny':<10}{'GPT-2':<12}{'TinyStories':<14}{-0.408:<10.3f}")
print(f"{'Tiny':<10}{'GPT-2':<12}{'WikiText-2':<14}{-0.206:<10.3f}")
print(f"{'Tiny':<10}{'GPT-Neo':<12}{'TinyStories':<14}{-0.376:<10.3f}")
print(f"{'Tiny':<10}{'GPT-Neo':<12}{'WikiText-2':<14}{wiki_neo_results['Tiny']['gap']:<10.3f}")
print()
print(f"{'Medium':<10}{'GPT-2':<12}{'TinyStories':<14}{-0.271:<10.3f}")
print(f"{'Medium':<10}{'GPT-2':<12}{'WikiText-2':<14}{-0.083:<10.3f}")
print(f"{'Medium':<10}{'GPT-Neo':<12}{'TinyStories':<14}{-0.231:<10.3f}")
print(f"{'Medium':<10}{'GPT-Neo':<12}{'WikiText-2':<14}{wiki_neo_results['Medium']['gap']:<10.3f}")

Size      Teacher     Dataset       Gap       
Tiny      GPT-2       TinyStories   -0.408    
Tiny      GPT-2       WikiText-2    -0.206    
Tiny      GPT-Neo     TinyStories   -0.376    
Tiny      GPT-Neo     WikiText-2    -0.145    

Medium    GPT-2       TinyStories   -0.271    
Medium    GPT-2       WikiText-2    -0.083    
Medium    GPT-Neo     TinyStories   -0.231    
Medium    GPT-Neo     WikiText-2    -0.110    
